In [0]:
df = spark.table("sales_data")

### Q1: Roles of Driver, Cluster Manager, and Executor

The **Driver** is the brain of the application — it runs the main code, plans the work, splits it into tasks, and collects the final results. If it crashes, the whole job stops.

The **Cluster Manager** (YARN, Kubernetes, Mesos, or Spark Standalone) doesn't process data at all — it just allocates available machines/resources to the Driver whenever it needs workers.

The **Executor** does the actual work. It runs the tasks assigned by the Driver — filtering, transforming, aggregating — on its share of the data, in parallel with other executors, and sends results back to the Driver.

### Q2: How does Lazy Evaluation improve performance?

Spark doesn't run transformations like `.filter()` or `.select()` immediately — it just builds a plan (DAG) and waits. Only when an **action** like `.show()` or `.count()` is called does it actually execute.

This lets Spark look at the **entire chain of steps together** and optimize them — combining operations into a single pass, skipping unused columns, and pushing filters earlier — instead of processing step-by-step inefficiently.

Spark plans first, then executes smartly in one optimized run, saving both memory and processing time on large datasets.

### Q3: Read CSV with header and inferSchema enabled

In [0]:
df = spark.read.csv("data/source.csv", header=True, inferSchema=True)

Here, `header=True` tells Spark that the first row contains column names, not actual data. `inferSchema=True` tells Spark to automatically detect the correct data type for each column (integer, double, string, etc.) instead of treating everything as plain text.

### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

**Answer:**

CSV is a **row-based** format — it stores data row by row, as plain text, one full record after another. Parquet is a **columnar** format — it stores data column by column, grouping all values of the same column together in a compressed binary form.

This matters for performance because most queries only need a few columns out of many. With CSV, Spark has to read every column of every row just to get one field — wasting a lot of I/O. With Parquet, Spark can go directly to the specific column needed and skip the rest entirely (column pruning), which also allows much better compression and much faster reads.

 CSV forces you to read the whole row to get one value; Parquet lets you read only the columns you actually need, making it far faster and lighter for big data analytics.

### Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [0]:
df = spark.table("sales_data")

In [0]:
df.filter(df.category == "Electronics") \
  .select("product_id", "price") \
  .show()

+----------+--------+
|product_id|   price|
+----------+--------+
|     P1001|66868.77|
|     P1002|53349.08|
|     P1003|50731.43|
|     P1005|32628.65|
|     P1006|87593.86|
|     P1008| 34521.3|
|     P1020|10512.16|
|     P1024|30758.66|
|     P1028|60766.13|
|     P1032|88685.14|
|     P1034|67443.81|
|     P1035|86489.71|
|     P1039|41516.55|
|     P1044| 8760.71|
|     P1046|85082.06|
|     P1048|32154.24|
|     P1052|87036.57|
|     P1053|10823.09|
|     P1062|22413.41|
|     P1064|70168.63|
+----------+--------+
only showing top 20 rows


Here, `.filter()` keeps only the rows where `category` equals `'Electronics'`, and `.select()` then picks out only the `product_id` and `price` columns from those filtered rows.

Applying this to our `sales_data`, this would return just the `product_id` and `price` for all Electronics items like Monitor, Router, Headphones, and AC.

### Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [0]:
df.withColumnRenamed("old_name", "new_name") \
  .withColumn("price", df["price"].cast("double")) \
  .show()

+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+
|user_id|product_id|  new_name|       category|   price|        base_price|    status|            amount|region|priority|
+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+
|   1163|     P1001|   Monitor|    Electronics|66868.77|66868.770000000000|   Pending|60957.930000000000| North|     Low|
|   1189|     P1002|    Router|    Electronics|53349.08|53349.080000000000| Completed|43474.090000000000| South|    High|
|   1129|     P1003|Headphones|    Electronics|50731.43|50731.430000000000|Processing|46176.780000000000|  East|    High|
|   1194|     P1004|     Mouse|    Accessories|30952.42|30952.420000000000|   Pending|28094.180000000000|  East|    High|
|   1023|     P1005|        AC|    Electronics|32628.65|32628.650000000000| Cancelled|35952.300000000000| North|     Low|
|   1117|     P1006|    

### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

**Answer:**

Every RDD/DataFrame in Spark keeps track of its **lineage** — the exact sequence of transformations used to build it, starting from the original data source. This chain of steps forms a **DAG (Directed Acyclic Graph)**.

Spark does not physically replicate data across nodes for fault tolerance. Instead, if a worker node fails and some partitions of data are lost, the Driver looks at the lineage graph, identifies exactly which transformations produced those lost partitions, and **recomputes only that missing part** on a healthy executor — using the original source data.

This makes recovery efficient, since Spark doesn't need extra storage for backups, and it only redoes the minimum necessary work rather than restarting the entire job. For very long lineage chains, `.cache()`/`.persist()` or checkpointing is used to avoid expensive recomputation.

### Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [0]:
df.filter(
    (df.status == "Completed") & (df.amount > 1000)
).show()

+-------+----------+--------+---------------+--------+------------------+---------+-------------------+------+--------+
|user_id|product_id|old_name|       category|   price|        base_price|   status|             amount|region|priority|
+-------+----------+--------+---------------+--------+------------------+---------+-------------------+------+--------+
|   1189|     P1002|  Router|    Electronics|53349.08|53349.080000000000|Completed| 43474.090000000000| South|    High|
|   1158|     P1007|     Bed|         Sports|17709.83|17709.830000000000|Completed| 14573.640000000000| South|  Medium|
|   1094|     P1009|     Bed|      Furniture|60480.68|60480.680000000000|Completed| 66804.870000000000| South|     Low|
|   1152|     P1016|   Mixer|         Sports|23001.56|23001.560000000000|Completed| 26225.040000000000| North|     Low|
|   1184|     P1018|    Desk|         Sports|68695.62|68695.620000000000|Completed| 84856.899999999990|  East|     Low|
|   1014|     P1021|  Camera|         Sp

`.filter()` keeps only rows where **both** conditions are true — `status == "Completed"` AND `amount > 1000`. The `&` operator combines them, with each condition in parentheses (required by Spark syntax). Writing it across multiple lines is just for readability.

### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

**Answer:**

Predicate Pushdown means Spark sends the filter condition down to the file-reading layer itself, instead of loading all the data first and filtering afterward.

Parquet files store data in **row groups**, and each row group keeps metadata with the **min/max values** of each column. When a filter like `year == 2024` is applied, Spark checks these min/max stats before reading — if a row group's range can't possibly contain 2024, that entire row group is **skipped and never loaded into memory**.

This drastically reduces disk I/O and memory usage, since only the row groups that could actually match the filter are read. This is not possible with CSV, since it has no such stored statistics, forcing Spark to read every row completely.

### Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [0]:
df.withColumn("final_price", df.base_price * 1.18) \
  .show()

+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+------------------+
|user_id|product_id|  old_name|       category|   price|        base_price|    status|            amount|region|priority|       final_price|
+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+------------------+
|   1163|     P1001|   Monitor|    Electronics|66868.77|66868.770000000000|   Pending|60957.930000000000| North|     Low|        78905.1486|
|   1189|     P1002|    Router|    Electronics|53349.08|53349.080000000000| Completed|43474.090000000000| South|    High|        62951.9144|
|   1129|     P1003|Headphones|    Electronics|50731.43|50731.430000000000|Processing|46176.780000000000|  East|    High|        59863.0874|
|   1194|     P1004|     Mouse|    Accessories|30952.42|30952.420000000000|   Pending|28094.180000000000|  East|    High|36523.855599999995|
|   1023|    

`.withColumn()` adds a new column called `final_price`, calculated by multiplying the existing `base_price` column by `1.18` (adding 18% tax) for every row.

### Q11: What is the difference between Transformations and Actions? Provide two examples of each.

**Answer:**

**Transformations** are lazy operations — they don't run immediately, they just build up a plan (DAG) describing what to do to the data. They return a new DataFrame.
- Examples: `.filter()`, `.select()`

**Actions** trigger the actual execution of everything planned so far, and return a real result (a number, a list, or a saved file).
- Examples: `.count()`, `.collect()`

Because transformations are lazy, Spark can look at the entire chain together and optimize it before running anything — combining steps, skipping unnecessary columns, and applying filters early — instead of executing each line immediately and wastefully.

### Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [0]:
df = spark.read.parquet("path/to/input")

df.filter(df.user_id.isNotNull()) \
  .write.csv("path/to/output", header=True)

First, `spark.read.parquet()` loads the Parquet file into a DataFrame. Then `.filter(df.user_id.isNotNull())` keeps only the rows where `user_id` is not null. Finally, `.write.csv()` saves the cleaned result as a CSV file, with `header=True` so column names are included in the output file.

### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

**Answer:**

The main difference is **where the Driver runs**.

In **Client Mode**, the Driver runs on the machine that submitted the job (like your laptop or an edge node) — outside the cluster. If that machine disconnects or shuts down, the job dies, since the Driver goes down with it. This mode is best for interactive work like debugging or notebooks.

In **Cluster Mode**, the Driver runs inside the cluster itself, on one of the worker nodes, managed by the Cluster Manager. This means the job keeps running even if the machine that submitted it disconnects — making it the preferred choice for production, scheduled jobs.

### Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [0]:
df.filter(
    (df.region == "North") | (df.priority == "High")
).show()

+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+
|user_id|product_id|  old_name|       category|   price|        base_price|    status|            amount|region|priority|
+-------+----------+----------+---------------+--------+------------------+----------+------------------+------+--------+
|   1163|     P1001|   Monitor|    Electronics|66868.77|66868.770000000000|   Pending|60957.930000000000| North|     Low|
|   1189|     P1002|    Router|    Electronics|53349.08|53349.080000000000| Completed|43474.090000000000| South|    High|
|   1129|     P1003|Headphones|    Electronics|50731.43|50731.430000000000|Processing|46176.780000000000|  East|    High|
|   1194|     P1004|     Mouse|    Accessories|30952.42|30952.420000000000|   Pending|28094.180000000000|  East|    High|
|   1023|     P1005|        AC|    Electronics|32628.65|32628.650000000000| Cancelled|35952.300000000000| North|     Low|
|   1020|     P1008|    

`.filter()` keeps rows where **either** condition is true — `region == "North"` OR `priority == "High"`. The `|` operator is used to combine conditions with OR logic, with each condition wrapped in parentheses.

### Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

**Answer:**

`.collect()` pulls **every single row** from all executors back into the Driver's memory as a local list. On a multi-terabyte dataset, the Driver almost certainly doesn't have enough memory to hold all of that, so it will crash with an `OutOfMemoryError`.

`.show(5)` only computes and returns a small sample (5 rows) — Spark's planner optimizes this so it typically only scans a few partitions rather than the entire dataset, and stops as soon as it has enough rows to display.

 use `.show()`, `.take(n)`, or `.limit(n)` for exploring large datasets, and reserve `.collect()` only for cases where the result is guaranteed to be small (e.g., after heavy aggregation).